# Phase 2 — Task 3: Cross-Stock Transfer (The Headline Metric)

**Protocol (from `phase2-downstream-probes.md`):**
* **Source Stock:** `sz000001` trained encoder (completely frozen).
* **Target Stocks:** The other 4 stocks (`sz000002`, `sz000858`, `sz300147`, `sz002415`).
* **Low-Resource Fine-Tuning:** Takes exactly **20% of target stock train split** to train a fresh `TrendHead`.
* **Evaluation:** Evaluated on the target stock's full test split.
* **Headline Metric:** **Mean Macro-F1** across all 4 target stocks.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive/JEPA_LOB/baselines')
!pip install -q lightning pandas numpy torch scikit-learn
print("✓ Environment and Google Drive ready.")


In [ ]:
import os, sys, time
import numpy as np
import pandas as pd
import torch
from downstream_common import (
    MODEL_REGISTRY, STOCKS, LATENT_DIM, set_seed,
    load_frozen_encoder, compute_trend_labels_and_windows,
    train_trend_head_probe
)

set_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
os.makedirs("downstream_results", exist_ok=True)
TARGET_STOCKS = ['sz000002', 'sz000858', 'sz300147', 'sz002415']


In [ ]:
# Run sz000001 -> Target Stocks Transfer Evaluation
transfer_results = []
print("=" * 80)
print("  TASK 3: CROSS-STOCK TRANSFER (Source: sz000001 -> 4 Target Stocks)")
print("=" * 80)

for model_name in MODEL_REGISTRY.keys():
    print(f"\nEvaluating Transfer for Model: {model_name} (Source: sz000001)")
    # Load frozen source encoder trained on sz000001
    source_encoder = load_frozen_encoder(model_name, stock='sz000001', device=device)
    
    for tgt_stock in TARGET_STOCKS:
        # Load target stock data & labels
        tgt_csv = f"data/{tgt_stock}-level10_processed.csv"
        df_tgt = pd.read_csv(tgt_csv)
        split_indices, split_labels, theta = compute_trend_labels_and_windows(df_tgt, k=5, seq_len=100)
        tgt_features = df_tgt.iloc[:, 1:].values
        
        # Subsample exactly 20% of target train split (first 20% temporally)
        n_train_full = len(split_indices['train'])
        n_train_20pct = int(n_train_full * 0.20)
        train_starts = split_indices['train'][:n_train_20pct]
        train_y = split_labels['train'][:n_train_20pct]
        
        val_starts = split_indices['val']
        val_y = split_labels['val']
        test_starts = split_indices['test']
        test_y = split_labels['test']
        
        # Extract latents using frozen source encoder
        def extract_z(starts):
            z_list = []
            for b in range(0, len(starts), 512):
                b_starts = starts[b : b + 512]
                b_win = np.stack([tgt_features[s : s + 100] for s in b_starts])
                b_t = torch.tensor(b_win, dtype=torch.float32, device=device)
                with torch.no_grad():
                    z_list.append(source_encoder(b_t).cpu().numpy())
            return np.concatenate(z_list, axis=0) if z_list else np.empty((0, LATENT_DIM))
            
        train_z = extract_z(train_starts)
        val_z   = extract_z(val_starts)
        test_z  = extract_z(test_starts)
        
        t0 = time.time()
        metrics = train_trend_head_probe(
            train_z, train_y, val_z, val_y, test_z, test_y,
            epochs=50, lr=1e-3, batch_size=256, device=device
        )
        elapsed = time.time() - t0
        
        print(f"  {model_name:<12} -> {tgt_stock}: Macro-F1 = {metrics['macro_f1']:.4f}, Acc = {metrics['accuracy']:.4f} (20% train samples: {len(train_z)}, {elapsed:.1f}s)")
        
        row = {
            'model': model_name,
            'source_stock': 'sz000001',
            'target_stock': tgt_stock,
            'transfer_macro_f1': metrics['macro_f1'],
            'transfer_accuracy': metrics['accuracy'],
            'prec_down': metrics['precision_down'],
            'rec_down': metrics['recall_down'],
            'prec_stable': metrics['precision_stable'],
            'rec_stable': metrics['recall_stable'],
            'prec_up': metrics['precision_up'],
            'rec_up': metrics['recall_up'],
            'theta': theta
        }
        transfer_results.append(row)

df_transfer = pd.DataFrame(transfer_results)
df_transfer.to_csv("downstream_results/transfer_results.csv", index=False)
print("\n✓ Saved: downstream_results/transfer_results.csv")


In [ ]:
# Headline Table: Cross-Stock Transfer Matrix & Headline Mean Macro-F1
pivot_tr = df_transfer.pivot(index='model', columns='target_stock', values='transfer_macro_f1')
pivot_tr['Headline Mean Macro-F1'] = pivot_tr.mean(axis=1)
pivot_tr = pivot_tr.sort_values(by='Headline Mean Macro-F1', ascending=False)

print("=" * 80)
print("  TASK 3 HEADLINE: CROSS-STOCK TRANSFER MACRO-F1 RANKING")
print("=" * 80)
display(pivot_tr.round(4))
